# House Price Prediction Project

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn import metrics
import pickle

## 1. Load Dataset

In [ ]:
data_url = "http://lib.stat.cmu.edu/datasets/boston"
raw_df = pd.read_csv(data_url, sep="\s+", skiprows=22, header=None)

# Combine alternating rows
data = np.hstack([raw_df.values[::2, :], raw_df.values[1::2, :2]])
target = raw_df.values[1::2, 2]

feature_names = ['CRIM', 'ZN', 'INDUS', 'CHAS', 'NOX', 'RM', 'AGE', 'DIS', 'RAD', 'TAX', 'PTRATIO', 'B', 'LSTAT']

df = pd.DataFrame(data, columns=feature_names)
df['price'] = target
df.head()

## 2. Data Cleaning and EDA

In [ ]:
# Dataset shape
print("Shape:", df.shape)

# Check for missing values
print("Null counts:")
print(df.isnull().sum())

In [ ]:
# Summary statistics
df.describe()

In [ ]:
# Correlation matrix
plt.figure(figsize=(10, 8))
sns.heatmap(df.corr(), annot=True, fmt=".1f", cmap="coolwarm", square=True)
plt.title("Correlation Matrix")
plt.show()

In [ ]:
# RM vs Price and LSTAT vs Price scatter plots
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
sns.scatterplot(x=df['RM'], y=df['price'])
plt.title('RM vs Price')
plt.xlabel('Rooms')
plt.ylabel('Price')

plt.subplot(1, 2, 2)
sns.scatterplot(x=df['LSTAT'], y=df['price'])
plt.title('LSTAT vs Price')
plt.xlabel('LSTAT')
plt.ylabel('Price')

plt.tight_layout()
plt.show()

## 3. Split Dataset

In [ ]:
X = df.drop(columns=['price'])
Y = df['price']

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=2)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

## 4. Model Training

In [ ]:
model = LinearRegression()
model.fit(X_train, Y_train)

## 5. Model Evaluation and Coefficients

In [ ]:
train_preds = model.predict(X_train)
test_preds = model.predict(X_test)

train_rmse = np.sqrt(metrics.mean_squared_error(Y_train, train_preds))
test_rmse = np.sqrt(metrics.mean_squared_error(Y_test, test_preds))

train_r2 = metrics.r2_score(Y_train, train_preds)
test_r2 = metrics.r2_score(Y_test, test_preds)

print("--- Train Metrics ---")
print(f"RMSE: {train_rmse:.3f}")
print(f"R2 Score: {train_r2:.3f}")

print("\n--- Test Metrics ---")
print(f"RMSE: {test_rmse:.3f}")
print(f"R2 Score: {test_r2:.3f}")

In [ ]:
print("Intercept:", model.intercept_)

coeff_df = pd.DataFrame(model.coef_, X.columns, columns=['Coefficient'])
coeff_df.sort_values(by='Coefficient', ascending=False)

## 6. Save Model and Inference

In [ ]:
with open('linear_regression_model.pkl', 'wb') as file:
    pickle.dump(model, file)

In [ ]:
with open('linear_regression_model.pkl', 'rb') as file:
    loaded_model = pickle.load(file)

sample_house = np.array([[0.006, 18.0, 2.31, 0.0, 0.538, 6.575, 65.2, 4.09, 1.0, 296.0, 15.3, 396.9, 4.98]])
pred = loaded_model.predict(sample_house)
print(f"Predicted Price: ${pred[0]:.2f}k (${pred[0]*1000:,.2f})")